# 03_Match — Relations Centre <-> Service (et autres relations métier)

**Correction majeure vs la version précédente** : l'ancien pipeline
matchait à 100% par institution puis **pruned après coup** (moyenne
19,2 services/centre, ~50% des relations impliquant EPS). Ici, le plafond de
**top-8 relations/centre** est appliqué **au moment du matching lui-même**,
pas en nettoyage a posteriori — cohérent avec le tableau de scoring du guide.

**Cascade de scoring Centre -> Service** :
1. Intersection d'institutions non-EPS -> **1.00** (`exact_institution`)
2. Intersection == {EPS} uniquement -> **0.50** (`eps_fallback`, pénalisé — EPS
   est un dernier recours, jamais un match "normal")
3. Aucune institution commune mais population cible du service mentionnée
   dans `centre.personnes_cibles` -> **0.75** (`population_commune`)
4. Rien -> pas de relation (mieux vaut aucune relation qu'une relation
   inventée)

Puis : conservation du **top-8 par centre** trié par score, avec un
**plafond de 2 relations EPS max par centre** — un centre de type EPS ne
doit pas voir ses 8 emplacements remplis uniquement de matches EPS à bas
score ; mieux vaut moins de relations mais pertinentes.

In [1]:
import sys
from pathlib import Path
from collections import defaultdict

sys.path.insert(0, str(Path.cwd()))
from etl_lib.ontology import POPULATION_CIBLES, normalize_arabic
from etl_lib.io_utils import load_jsonl, save_jsonl

PROCESSED_DIR = Path.cwd().parent / "data" / "processed"
RELATIONS_DIR = Path.cwd().parent / "data" / "relations"
RELATIONS_DIR.mkdir(parents=True, exist_ok=True)

services = load_jsonl(PROCESSED_DIR / "services_unified.jsonl")
centres = load_jsonl(PROCESSED_DIR / "centres_normalized.jsonl")
programmes = load_jsonl(PROCESSED_DIR / "programmes_2027.jsonl")
faqs = load_jsonl(PROCESSED_DIR / "faq_aos.jsonl")
print(f"services={len(services)} centres={len(centres)} programmes={len(programmes)} faq={len(faqs)}")


services=61 centres=3328 programmes=7 faq=8


## 1. Relations Centre -> Service (cascade + plafond top-8)

In [2]:
TOP_N_PER_CENTRE = 8
MAX_EPS_PER_CENTRE = 2  # EPS = dernier recours : jamais plus de 2/8 relations

# === FIX_POPULATION_COMMUNE_2026_08_14 ===
# centres_normalized.jsonl:personnes_cibles n'a QUE 5 valeurs distinctes
# (verifie sur les 3328 centres), en francais abrege -- le check generique
# par sous-chaine (slug categorie ou libelle complet AR/FR) ne matchait
# JAMAIS "Pers. en sit . hand." ni "Pers. âgés en sit. diff.", laissant
# personnes_handicapees (372 centres taggues) et personnes_agees (66
# centres) a 0 relation Centre->Service malgre des donnees population
# explicites (trouve en audit le 2026-08-14). Mapping explicite plutot que
# d'elargir le check generique : les valeurs sont fixes et connues (5 au
# total), un mapping exact est plus fiable qu'un matching flou supplementaire.
_PERSONNES_CIBLES_MAP = {
    "enfants en sit. diff.": "enfants",
    "femmes en sit. diff.": "femmes",
    "pers. en sit . hand.": "personnes_handicapees",
    "pers. âgés en sit. diff.": "personnes_agees",
    # "tc" (351 centres) : signification ambigue dans la source (non
    # documentee), laisse sans mapping plutot que de deviner -- ces centres
    # restent eligibles via exact_institution/eps_fallback normalement.
}


def score_centre_service(centre: dict, service: dict):
    c_inst = set(centre["institutions"])
    s_inst = set(service["institutions"])
    common = c_inst & s_inst

    if common - {"EPS"}:
        return 1.00, "exact_institution"

    pop_text = normalize_arabic(centre.get("personnes_cibles", ""))
    mapped_category = _PERSONNES_CIBLES_MAP.get(pop_text)
    if mapped_category and mapped_category == service["categorie"]:
        return 0.75, "population_commune"

    service_pop = POPULATION_CIBLES.get(service["categorie"], {})
    if pop_text and any(normalize_arabic(kw) in pop_text for kw in [service["categorie"], service_pop.get("ar", ""), service_pop.get("fr", "")] if kw):
        return 0.75, "population_commune"

    if common == {"EPS"}:
        return 0.50, "eps_fallback"

    return 0.0, None


services_by_population = defaultdict(list)
for s in services:
    services_by_population[s["categorie"]].append(s)

relations_c2s = []
centres_sans_match = 0
for centre in centres:
    candidates = []
    # on ne compare qu'aux services de la meme population cible + tous les
    # services dont l'institution est explicitement partagee (evite de
    # comparer un centre "enfants" a un service "personnes agees" sans lien)
    pool = list(services)
    for service in pool:
        score, method = score_centre_service(centre, service)
        if score > 0:
            candidates.append((service["id"], score, method))

# === FIX_TIEBREAK_2026_08_14 ===
    # Depart les egalites par hash deterministe (centre_id, service_id) --
    # sans ca, des centaines de centres a score identique (0.75) selectionnent
    # tous les 8 PREMIERS services du fichier (ordre stable), laissant les
    # suivants orphelins pour toujours. Deterministe -> idempotent (memes
    # entrees, meme resultat a chaque execution), contrairement a un tri
    # aleatoire non reproductible.
    import hashlib as _hashlib

    def _tie_key(service_id):
        return _hashlib.md5(f"{centre['id']}_{service_id}".encode()).hexdigest()

    candidates.sort(key=lambda x: (x[1], _tie_key(x[0])), reverse=True)
    # Plafond top-8 global, MAIS on ne remplit pas les places restantes en
    # forcant des matches EPS a bas score juste pour atteindre 8 : EPS est
    # un dernier recours, pas une case a remplir. Max MAX_EPS_PER_CENTRE
    # relations EPS par centre, quitte a ce qu'un centre EPS-type finisse
    # avec moins de 8 relations au total.
    top, n_eps = [], 0
    for service_id, score, method in candidates:
        if len(top) >= TOP_N_PER_CENTRE:
            break
        if method == "eps_fallback":
            if n_eps >= MAX_EPS_PER_CENTRE:
                continue
            n_eps += 1
        top.append((service_id, score, method))
    if not top:
        centres_sans_match += 1
    for service_id, score, method in top:
        relations_c2s.append({
            "source_id": centre["id"], "source_type": "Centre",
            "target_id": service_id, "target_type": "Service",
            "score": score, "method": method,
        })

save_jsonl(RELATIONS_DIR / "relations_centre_service.jsonl", relations_c2s)

n_centres_avec_relation = len(centres) - centres_sans_match
avg_per_centre = len(relations_c2s) / max(n_centres_avec_relation, 1)
eps_relations = sum(1 for r in relations_c2s if r["method"] == "eps_fallback")
eps_pct = 100 * eps_relations / max(len(relations_c2s), 1)

print(f"Relations Centre->Service : {len(relations_c2s)}")
print(f"Centres sans aucun match  : {centres_sans_match}/{len(centres)}")
print(f"Moyenne services/centre (parmi les centres matches) : {avg_per_centre:.1f} (cible guide : <= 8)")
print(f"Relations EPS fallback    : {eps_relations} ({eps_pct:.1f}%, cible guide : < 20%)")


Relations Centre->Service : 24518
Centres sans aucun match  : 0/3328
Moyenne services/centre (parmi les centres matches) : 7.4 (cible guide : <= 8)
Relations EPS fallback    : 702 (2.9%, cible guide : < 20%)


## 2. Autres relations métier

In [3]:
# Service -> PopulationCible (deja connu via services['categorie'])
relations_s2p = [
    {"source_id": s["id"], "source_type": "Service", "target_id": s["categorie"], "target_type": "PopulationCible"}
    for s in services if s["categorie"] in POPULATION_CIBLES
]
save_jsonl(RELATIONS_DIR / "relations_service_population.jsonl", relations_s2p)

# Programme -> Institution / PopulationCible
relations_p2i, relations_p2p = [], []
for p in programmes:
    for inst in p.get("institutions_liees", []):
        relations_p2i.append({"source_id": p["id"], "source_type": "Programme", "target_id": inst, "target_type": "Institution"})
    pop_text = normalize_arabic(p.get("population_cible", ""))
    for pop_code in POPULATION_CIBLES:
        if pop_code in pop_text:
            relations_p2p.append({"source_id": p["id"], "source_type": "Programme", "target_id": pop_code, "target_type": "PopulationCible"})
save_jsonl(RELATIONS_DIR / "relations_programme_institution.jsonl", relations_p2i)
save_jsonl(RELATIONS_DIR / "relations_programme_population.jsonl", relations_p2p)

# Programme -> Institution => derive Centre -> Programme (un centre dont une
# institution est liee au programme "beneficie" potentiellement du programme)
relations_c2prog = []
prog_by_institution = defaultdict(list)
for p in programmes:
    for inst in p.get("institutions_liees", []):
        prog_by_institution[inst].append(p["id"])
for centre in centres:
    for inst in centre["institutions"]:
        for prog_id in prog_by_institution.get(inst, []):
            relations_c2prog.append({"source_id": centre["id"], "source_type": "Centre", "target_id": prog_id, "target_type": "Programme"})
save_jsonl(RELATIONS_DIR / "relations_centre_programme.jsonl", relations_c2prog)

# FAQ -> Programme (REPOND_A) uniquement si code programme identifie explicitement
relations_f2p = []
prog_codes = {p["code"] for p in programmes}
for f in faqs:
    if f.get("programme") in prog_codes:
        relations_f2p.append({"source_id": f["id"], "source_type": "FAQ", "target_id": f["programme"], "target_type": "Programme"})
save_jsonl(RELATIONS_DIR / "relations_faq_programme.jsonl", relations_f2p)

print(f"Service->PopulationCible  : {len(relations_s2p)}")
print(f"Programme->Institution    : {len(relations_p2i)}")
print(f"Programme->PopulationCible: {len(relations_p2p)}")
print(f"Centre->Programme (derive): {len(relations_c2prog)}")
print(f"FAQ->Programme (REPOND_A) : {len(relations_f2p)}")


Service->PopulationCible  : 61
Programme->Institution    : 13
Programme->PopulationCible: 8
Centre->Programme (derive): 9573
FAQ->Programme (REPOND_A) : 0


## 3. Rapport de qualité du matching

In [4]:
import json as _json
report = {
    "relations_centre_service": {
        "total": len(relations_c2s),
        "centres_matches": n_centres_avec_relation,
        "centres_sans_match": centres_sans_match,
        "moyenne_par_centre": round(avg_per_centre, 2),
        "eps_fallback_pct": round(eps_pct, 1),
        "methodes": {m: sum(1 for r in relations_c2s if r["method"] == m) for m in ("exact_institution", "eps_fallback", "population_commune")},
    },
    "autres_relations": {
        "service_population": len(relations_s2p), "programme_institution": len(relations_p2i),
        "programme_population": len(relations_p2p), "centre_programme": len(relations_c2prog),
        "faq_programme": len(relations_f2p),
    },
}
(Path.cwd().parent / "data" / "reports" / "report_03_match.json").write_text(
    _json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print(_json.dumps(report, ensure_ascii=False, indent=2))
print("\n\u2705 03_Match termine — biais EPS corrige a la source (plafond top-8 applique au matching, pas en nettoyage a posteriori).")


{
  "relations_centre_service": {
    "total": 24518,
    "centres_matches": 3328,
    "centres_sans_match": 0,
    "moyenne_par_centre": 7.37,
    "eps_fallback_pct": 2.9,
    "methodes": {
      "exact_institution": 3329,
      "eps_fallback": 702,
      "population_commune": 20487
    }
  },
  "autres_relations": {
    "service_population": 61,
    "programme_institution": 13,
    "programme_population": 8,
    "centre_programme": 9573,
    "faq_programme": 0
  }
}

✅ 03_Match termine — biais EPS corrige a la source (plafond top-8 applique au matching, pas en nettoyage a posteriori).
